# 01. Exploratory Data Analysis (EDA)
## DSN Bootcamp Challenge: Sales Forecasting

**Objective**: Understand the data structure, distributions, relationships, and quality issues

**Evaluation Metric**: RMSE (Root Mean Squared Error)

---

## 1. Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Import utility functions
import sys
sys.path.append('../')
from src.utils import set_seed, print_seed_info, get_data_summary, rmse, mae

# Set random seed for reproducibility
set_seed(42)
print_seed_info(42)

## 2. Load Data

In [ ]:
# Load training and test datasets
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns: {list(test_df.columns)}")

## 3. Data Overview & Schema Validation

In [ ]:
# Display first few rows
print("\n=== TRAIN DATA (First 5 rows) ===")
print(train_df.head())

print("\n=== TEST DATA (First 5 rows) ===")
print(test_df.head())

In [ ]:
# Data types and schema
print("\n=== DATA TYPES ===")
print(f"\nTrain:")
print(train_df.dtypes)
print(f"\nTest:")
print(test_df.dtypes)

## 4. Missing Values Analysis

In [ ]:
# Missing values in train set
print("\n=== MISSING VALUES (Train) ===")
missing_train = train_df.isnull().sum()
missing_train_pct = (missing_train / len(train_df)) * 100
missing_summary = pd.DataFrame({
    'Column': missing_train.index,
    'Missing_Count': missing_train.values,
    'Missing_Percent': missing_train_pct.values
}).sort_values('Missing_Count', ascending=False)

print(missing_summary[missing_summary['Missing_Count'] > 0])

# Missing values in test set
print("\n=== MISSING VALUES (Test) ===")
missing_test = test_df.isnull().sum()
missing_test_pct = (missing_test / len(test_df)) * 100
missing_summary_test = pd.DataFrame({
    'Column': missing_test.index,
    'Missing_Count': missing_test.values,
    'Missing_Percent': missing_test_pct.values
}).sort_values('Missing_Count', ascending=False)

print(missing_summary_test[missing_summary_test['Missing_Count'] > 0])

## 5. Target Variable Analysis (total_sales)

In [ ]:
# Target statistics
print("\n=== TARGET VARIABLE STATISTICS (total_sales) ===")
print(train_df['total_sales'].describe())
print(f"\nSkewness: {train_df['total_sales'].skew():.4f}")
print(f"Kurtosis: {train_df['total_sales'].kurtosis():.4f}")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(train_df['total_sales'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Total Sales')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Total Sales')
axes[0].grid(alpha=0.3)

# KDE plot
train_df['total_sales'].plot(kind='kde', ax=axes[1], color='blue')
axes[1].set_xlabel('Total Sales')
axes[1].set_title('KDE Plot of Total Sales')
axes[1].grid(alpha=0.3)

# Box plot
axes[2].boxplot(train_df['total_sales'])
axes[2].set_ylabel('Total Sales')
axes[2].set_title('Box Plot of Total Sales')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../notebooks/target_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Target distribution plot saved")

## 6. Numerical Features Analysis

In [ ]:
# Identify numerical columns
numerical_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
# Remove ID and target
if 'id' in numerical_cols:
    numerical_cols.remove('id')
if 'total_sales' in numerical_cols:
    numerical_cols.remove('total_sales')

print(f"Numerical columns: {numerical_cols}")
print(f"\n=== NUMERICAL FEATURES STATISTICS ===")
print(train_df[numerical_cols].describe())

In [ ]:
# Visualize numerical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols[:4]):
    axes[idx].hist(train_df[col].dropna(), bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution: {col}')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/numerical_features_dist.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Numerical features distribution plot saved")

## 7. Categorical Features Analysis

In [ ]:
# Identify categorical columns
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if 'id' in categorical_cols:
    categorical_cols.remove('id')

print(f"Categorical columns: {categorical_cols}")
print(f"\n=== CATEGORICAL FEATURES VALUE COUNTS ===")
for col in categorical_cols:
    print(f"\n{col} ({train_df[col].nunique()} unique values):")
    print(train_df[col].value_counts().head(10))

In [ ]:
# Visualize categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(categorical_cols[:4]):
    train_df[col].value_counts().plot(kind='bar', ax=axes[idx], color='steelblue', edgecolor='black')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].set_title(f'Value Counts: {col}')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../notebooks/categorical_features_dist.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Categorical features distribution plot saved")

## 8. Store & Product Analysis

In [ ]:
# Store analysis
print("\n=== STORE ANALYSIS ===")
print(f"Unique stores: {train_df['store_code'].nunique()}")
print(f"\nStore format distribution:")
print(train_df['store_format'].value_counts())
print(f"\nStore size distribution:")
print(train_df['store_size'].value_counts())
print(f"\nStore location tier distribution:")
print(train_df['store_location_tier'].value_counts())

In [ ]:
# Product analysis
print("\n=== PRODUCT ANALYSIS ===")
print(f"Unique products: {train_df['product_code'].nunique()}")
print(f"\nProduct category distribution:")
print(train_df['product_category'].value_counts())
print(f"\nFat content distribution:")
print(train_df['fat_content'].value_counts())

## 9. Feature-Target Correlations

In [ ]:
# Numerical correlations with target
print("\n=== CORRELATION WITH TARGET (total_sales) ===")
correlation_with_target = train_df[numerical_cols + ['total_sales']].corr()['total_sales'].sort_values(ascending=False)
print(correlation_with_target)

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = train_df[numerical_cols + ['total_sales']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, 
            cbar_kws={'label': 'Correlation'}, square=True)
ax.set_title('Correlation Matrix: Numerical Features & Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Correlation heatmap saved")

## 10. Sales by Category Analysis

In [ ]:
# Sales by store type
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Store format
train_df.groupby('store_format')['total_sales'].agg(['mean', 'median', 'std']).plot(kind='bar', ax=axes[0])
axes[0].set_xlabel('Store Format')
axes[0].set_ylabel('Sales')
axes[0].set_title('Sales by Store Format')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3, axis='y')

# Store size
train_df.groupby('store_size')['total_sales'].agg(['mean', 'median']).plot(kind='bar', ax=axes[1])
axes[1].set_xlabel('Store Size')
axes[1].set_ylabel('Sales')
axes[1].set_title('Sales by Store Size')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3, axis='y')

# Store tier
train_df.groupby('store_location_tier')['total_sales'].agg(['mean', 'median']).plot(kind='bar', ax=axes[2])
axes[2].set_xlabel('Location Tier')
axes[2].set_ylabel('Sales')
axes[2].set_title('Sales by Store Location Tier')
axes[2].tick_params(axis='x', rotation=0)
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../notebooks/sales_by_category.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Sales by category plot saved")

In [ ]:
# Sales by product category
fig, ax = plt.subplots(figsize=(14, 6))
train_df.groupby('product_category')['total_sales'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False).head(15).plot(kind='barh', ax=ax)
ax.set_xlabel('Sales')
ax.set_title('Top 15 Product Categories by Average Sales')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../notebooks/sales_by_product_category.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Sales by product category plot saved")

## 11. Data Quality & Outliers

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), len(outliers) / len(data) * 100, lower_bound, upper_bound

print("\n=== OUTLIER DETECTION (IQR Method) ===")
print(f"{'Column':<20} {'Outlier_Count':<15} {'Outlier_Percent':<15} {'Lower_Bound':<15} {'Upper_Bound':<15}")
print("-" * 80)

for col in numerical_cols + ['total_sales']:
    count, pct, lb, ub = detect_outliers_iqr(train_df, col)
    print(f"{col:<20} {count:<15} {pct:<15.2f} {lb:<15.2f} {ub:<15.2f}")

In [ ]:
# Check for duplicates
print("\n=== DUPLICATE RECORDS ===")
print(f"Total duplicates (all columns): {train_df.duplicated().sum()}")
print(f"Duplicates (excluding ID): {train_df.duplicated(subset=[col for col in train_df.columns if col != 'id']).sum()}")

## 12. Summary & Insights

In [ ]:
print("\n" + "="*80)
print("EXPLORATORY DATA ANALYSIS - SUMMARY")
print("="*80)

print(f"\n📊 DATASET INFORMATION:")
print(f"  • Training samples: {len(train_df):,}")
print(f"  • Test samples: {len(test_df):,}")
print(f"  • Total features: {len(train_df.columns) - 1}  (excluding target)")
print(f"  • Numerical features: {len(numerical_cols)}")
print(f"  • Categorical features: {len(categorical_cols)}")

print(f"\n🎯 TARGET VARIABLE (total_sales):")
print(f"  • Mean: {train_df['total_sales'].mean():.2f}")
print(f"  • Median: {train_df['total_sales'].median():.2f}")
print(f"  • Std Dev: {train_df['total_sales'].std():.2f}")
print(f"  • Min: {train_df['total_sales'].min():.2f}")
print(f"  • Max: {train_df['total_sales'].max():.2f}")
print(f"  • Skewness: {train_df['total_sales'].skew():.4f}")

print(f"\n⚠️  DATA QUALITY ISSUES:")
print(f"  • Missing values in train: {train_df.isnull().sum().sum()} cells")
print(f"  • Missing values in test: {test_df.isnull().sum().sum()} cells")
print(f"  • Columns with missing data (train): {missing_train[missing_train > 0].index.tolist()}")
print(f"  • Duplicate rows: {train_df.duplicated().sum()}")

print(f"\n🏪 BUSINESS ENTITIES:")
print(f"  • Unique stores: {train_df['store_code'].nunique()}")
print(f"  • Unique products: {train_df['product_code'].nunique()}")
print(f"  • Store formats: {train_df['store_format'].nunique()}")
print(f"  • Product categories: {train_df['product_category'].nunique()}")

print(f"\n📈 TOP CORRELATIONS WITH TARGET:")
for i, (col, corr) in enumerate(correlation_with_target.head(6).items(), 1):
    if col != 'total_sales':
        print(f"  {i}. {col}: {corr:.4f}")

print(f"\n✅ RECOMMENDATIONS FOR NEXT STEPS:")
print(f"  1. Handle missing values in: {missing_train[missing_train > 0].index.tolist()}")
print(f"     → Use forward-fill for store_age, median for product_weight")
print(f"  2. Standardize categorical encoding (case sensitivity in product_category)")
print(f"  3. Create store/product aggregated features (mean sales, variance)")
print(f"  4. Generate lag features for time-series modeling")
print(f"  5. Apply target encoding for high-cardinality categoricals")
print(f"  6. Use time-based CV to prevent data leakage")

print("\n" + "="*80)

## 13. Data Export for Next Milestone

In [ ]:
# Save cleaned/preprocessed data summaries (for reference)
print("\n✓ EDA Complete!")
print("\nNext Steps:")
print("  1. Proceed to Milestone 3: Feature Engineering")
print("  2. Handle missing values and create lag/rolling features")
print("  3. Implement time-series cross-validation")
print("  4. Train baseline LightGBM model")